# LLM 수동 입력 테스트

저장소 루트에서 `uv run --locked jupyter lab ai/app/test.ipynb`로 엽니다.
분류 모델을 사용하지 않습니다. 분야와 질문을 직접 입력합니다.
실제 EXAONE 가중치와 세 어댑터가 필요하며, 최초 실행 시 모델을 다운로드할 수 있습니다.
CPU는 메모리와 생성 시간이 많이 필요합니다. 장치 설정을 바꾼 뒤에는 커널을 재시작하세요.

In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter

# CPU로 강제하려면 아래 두 줄의 주석을 해제하세요.
# os.environ['LLM_DEVICE'] = 'cpu'
# os.environ['LLM_LOAD_IN_4BIT'] = 'false'

for parent in (Path.cwd(), *Path.cwd().parents):
    ai_root = parent if (parent / 'app' / 'llm').is_dir() else parent / 'ai'
    if (ai_root / 'app' / 'llm').is_dir():
        sys.path.insert(0, str(ai_root))
        break
else:
    raise FileNotFoundError('dontalk 저장소 안에서 노트북을 실행하세요.')

from app.llm import config
from app.llm.model import answer, load_model

print(f'설정: device={config.LLM_DEVICE}, 4bit={config.LLM_LOAD_IN_4BIT}')

In [ ]:
started = perf_counter()
model, tokenizer = load_model()
print(f'모델 준비: {perf_counter() - started:.2f}초')
print(f'실제 장치: {model.get_input_embeddings().weight.device}')
assert load_model()[0] is model  # 같은 커널에서는 재로드하지 않습니다.

In [ ]:
MAX_NEW_TOKENS = 512

while True:
    category = input('분야 [은행/보험/증권], 끝내려면 종료: ').strip()
    if category == '종료':
        break
    question = input('질문: ').strip()
    if question == '종료':
        break
    started = perf_counter()
    try:
        result = answer(category, question, max_new_tokens=MAX_NEW_TOKENS)
    except ValueError as error:
        print(error)
        continue
    print(f"\n[{category}] {result}")
    print(f'답변 생성: {perf_counter() - started:.2f}초\n')